# SD 特征与跟踪

使用官方 `fishe8` 样本演示 Harris、Shi-Tomasi、HOG、128 维多尺度描述子和 `Feature_tracker_sd`。首次运行会下载测试数据；已有本地数据时，可将下面的 `sample_dir` 改为样本目录。角点检测返回 `(row, column)`，描述子输入使用 `(x, y, level)`。

In [ ]:
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import torch

from tianmoucv.data import TianmoucDataReader
from tianmoucv.isp import SD2XY
from tianmoucv.proc.features import (
    HarrisCorner, TomasiCorner, hog, steadyHarrisCornerForSIFT,
    compute_multiscale_sift_descriptor,
)
from tianmoucv.proc.tracking import Feature_tracker_sd, feature_matching
from tianmoucv.utils import ensure_tutorial_dataset, save_video

sample_dir = ensure_tutorial_dataset().sample_dir
reader = TianmoucDataReader(str(sample_dir), N=1, MAXLEN=3, print_info=False)
if len(reader) == 0:
    raise RuntimeError(f'样本中没有可读取的数据：{sample_dir}')
sample_key, relative_index = reader.locateSample(0)
raw_diff = reader.packRead(relative_index, sample_key, needPreProcess=False)['rawDiff']
print('样本数:', len(reader), '首个样本 AOP 帧数:', raw_diff.shape[1])

## 检测和匹配

SD 先经 `SD2XY` 转成图像梯度。这里的 128 维描述子是从梯度构造的 SIFT 风格描述子；下方跟踪类使用 OpenCV Shi-Tomasi 角点与金字塔 LK 光流。

In [ ]:
grad_x, grad_y = SD2XY(raw_diff[1:, 0].permute(1, 2, 0))
magnitude = torch.hypot(grad_x, grad_y)
harris_points = HarrisCorner(grad_x, grad_y, thresh=0.05, k=0.05)
tomasi_points = TomasiCorner(grad_x, grad_y, index=100)
hog_points, hog_descriptors = hog(
    grad_x, grad_y, [(column, row) for row, column in tomasi_points]
)
stable_points = steadyHarrisCornerForSIFT(
    grad_x, grad_y, num_levels=4, thresh=0.05, k=0.05
)
descriptors, descriptor_points = compute_multiscale_sift_descriptor(
    grad_x, grad_y, stable_points, num_levels=4
)

next_grad_x, next_grad_y = SD2XY(raw_diff[1:, 1].permute(1, 2, 0))
next_points = steadyHarrisCornerForSIFT(
    next_grad_x, next_grad_y, num_levels=4, thresh=0.05, k=0.05
)
next_descriptors, _ = compute_multiscale_sift_descriptor(
    next_grad_x, next_grad_y, next_points, num_levels=4
)
matches = feature_matching(descriptors, next_descriptors, ratio=0.8)
print('Harris:', len(harris_points), 'Tomasi:', len(tomasi_points))
print('HOG:', len(hog_descriptors), '维度:', len(hog_descriptors[0]) if hog_descriptors else 0)
print('多尺度描述子:', len(descriptors), '维度:', len(descriptors[0]) if descriptors else 0)
print('相邻帧匹配:', len(matches))

plt.figure(figsize=(10, 5))
plt.imshow(magnitude.numpy(), cmap='gray')
plt.scatter([column for row, column in harris_points],
            [row for row, column in harris_points], s=8, c='red')
plt.title('SD 梯度上的 Harris 角点')
plt.axis('off')
plt.show()

## 逐帧跟踪

只读取 `rawDiff`，避免为跟踪额外计算 RGB ISP、上采样和 HDR。输出视频以 30 fps 保存，便于查看；打印的跟踪速度统计 `update`（含轨迹叠加），不包含解码和 Matplotlib 绘图。

In [ ]:
tracker = Feature_tracker_sd(
    max_features=150, max_history=100, max_trail_age=100,
    redetect_interval=5, fb_threshold=1.5
)
frames = []
tracking_seconds = 0.0
for sample_index in range(min(3, len(reader))):
    sample_key, relative_index = reader.locateSample(sample_index)
    sample = reader.packRead(relative_index, sample_key, needPreProcess=False)
    raw_diff = sample['rawDiff']
    for frame_index in range(raw_diff.shape[1]):
        grad_x, grad_y = SD2XY(raw_diff[1:, frame_index].permute(1, 2, 0))
        magnitude = torch.hypot(grad_x, grad_y)
        started = perf_counter()
        _, visualization = tracker.update(magnitude, viz=True)
        tracking_seconds += perf_counter() - started
        frames.append(visualization)

video_path = save_video(frames, Path('feature_tracking.mp4'), fps=30, rgb_input=False)
print(f'跟踪 {len(frames)} 帧，update 平均速度：{len(frames) / tracking_seconds:.1f} fps')
print('视频:', video_path)
plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()